<div style="background:linear-gradient(135deg,#1e1b4b 0%,#4338ca 55%,#6366f1 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#c7d2fe;font-weight:700;text-transform:uppercase">Chapter 81 · Hypothesis Testing &amp; Inference</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">Choosing the Right Test 🧭</div>
  <div style="font-size:15px;color:#eef2ff;max-width:740px;line-height:1.6">You now own a toolbox of tests, the skill is picking the right one. Four questions (outcome type, number of groups, paired or not, assumptions) route every problem to its test. We run one mixed clinic dataset through the whole decision map.</div>
  <div style="margin-top:16px;font-size:13px;color:#c7d2fe">Statistics, Data Science and AI: A Visual Handbook · John Fisher · 2026</div>
</div>

## ⚙️ Setup

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
IND="#4f46e5"; DEEP="#4338ca"; LIGHT="#818cf8"; INK="#1a2138"; GRID="#e6e9f2"; GREEN="#059669"; RED="#ef4444"; AMBER="#d97706"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","figure.dpi":110,"font.size":11,
   "axes.edgecolor":GRID,"axes.grid":True,"grid.color":GRID,"axes.axisbelow":True,"axes.spines.top":False,
   "axes.spines.right":False,"axes.titlesize":12,"axes.titleweight":"bold","legend.frameon":False})
BASE="https://raw.githubusercontent.com/johnfisher-ai/Statistics-Data-Science-AI-Visual-Book/main/data/"
rng = np.random.default_rng(81)

<div style="background:#eef2ff;border-left:5px solid #4f46e5;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#4338ca;letter-spacing:1px">DEMO 1 · THE FOUR QUESTIONS</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">A decision tree, not a memory test</div>
<div style="color:#4a5578;margin-top:6px">Choosing a test is mechanical once you answer four questions: (1) Is the OUTCOME numeric or categorical? (2) How many GROUPS are you comparing (one, two, 3+)? (3) Are the groups INDEPENDENT or PAIRED? (4) Do the ASSUMPTIONS (rough normality, expected counts >= 5) hold, or do you need a rank-based test?</div>
</div>

In [2]:
guide = pd.DataFrame([
  ["numeric","1","-","one-sample t-test"],
  ["numeric","2","independent","two-sample (Welch) t-test  /  Mann-Whitney if skewed"],
  ["numeric","2","paired","paired t-test  /  Wilcoxon signed-rank if skewed"],
  ["numeric","3+","independent","one-way ANOVA  /  Kruskal-Wallis if skewed"],
  ["proportion","1","-","one-proportion z-test"],
  ["proportion","2","independent","two-proportion z-test  /  chi-square"],
  ["categorical","2+ vars","-","chi-square test of independence"],
], columns=["outcome","groups","design","test"])
print(guide.to_string(index=False))

    outcome  groups      design                                                 test
    numeric       1           -                                    one-sample t-test
    numeric       2 independent two-sample (Welch) t-test  /  Mann-Whitney if skewed
    numeric       2      paired     paired t-test  /  Wilcoxon signed-rank if skewed
    numeric      3+ independent           one-way ANOVA  /  Kruskal-Wallis if skewed
 proportion       1           -                                one-proportion z-test
 proportion       2 independent                 two-proportion z-test  /  chi-square
categorical 2+ vars           -                      chi-square test of independence


That is the entire chapter in one table. The rest is practice: read the question, identify the outcome type and the group structure, check the assumptions, and the test picks itself. We will run the clinic dataset through it.

<div style="background:#eef2ff;border-left:5px solid #4f46e5;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#4338ca;letter-spacing:1px">DEMO 2 · NUMERIC OUTCOMES</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Paired vs two-group vs many-group</div>
<div style="color:#4a5578;margin-top:6px">Load the clinic data. Numeric outcomes split three ways. Before/after on the SAME patient is PAIRED (paired t). Two independent arms is a two-sample t. Three clinics is ANOVA. Skew pushes each toward its rank-based twin.</div>
</div>

In [3]:
try:    d = pd.read_excel("../../data/ch81_clinic_visits.xlsx", sheet_name="Visits")
except FileNotFoundError: d = pd.read_excel(BASE+"ch81_clinic_visits.xlsx", sheet_name="Visits")
print("loaded:", d.shape)
# Q: did blood pressure drop within patients? PAIRED numeric -> paired t-test
pt = stats.ttest_rel(d.sysbp_after, d.sysbp_before)
print(f"[paired t] mean BP drop = {(d.sysbp_before-d.sysbp_after).mean():.2f} mmHg, t={pt.statistic:.2f}, p={pt.pvalue:.2e}")
# Q: does the NEW treatment drop BP more than standard? TWO independent groups -> Welch t
drop=d.sysbp_before-d.sysbp_after
new=drop[d.treatment=="new"]; std=drop[d.treatment=="standard"]
tt=stats.ttest_ind(new,std,equal_var=False)
print(f"[two-sample t] new drop {new.mean():.2f} vs standard {std.mean():.2f}, t={tt.statistic:.2f}, p={tt.pvalue:.2e}")

loaded: (200, 7)
[paired t] mean BP drop = 8.96 mmHg, t=-19.54, p=3.68e-48
[two-sample t] new drop 11.57 vs standard 6.26, t=6.35, p=1.49e-09


Same dataset, two numeric questions, two different t-tests, chosen purely from the design: <em>within</em>-patient change is paired; <em>between</em>-treatment comparison is independent two-sample. Misreading paired data as independent (Chapter 77) is the classic mistake this framework prevents.

<div style="background:#eef2ff;border-left:5px solid #4f46e5;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#4338ca;letter-spacing:1px">DEMO 3 · CATEGORICAL & MANY-GROUP</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Proportions, chi-square, and ANOVA</div>
<div style="color:#4a5578;margin-top:6px">A categorical outcome (satisfied yes/no by treatment) is a chi-square / two-proportion question. A numeric outcome across 3+ groups (wait time by clinic) is ANOVA, or Kruskal-Wallis if skewed. The framework routes each automatically.</div>
</div>

In [4]:
# Q: is satisfaction associated with treatment? TWO categoricals -> chi-square
ct=pd.crosstab(d.treatment, d.satisfied)
chi2,p,dof,_=stats.chi2_contingency(ct)
print(f"[chi-square] treatment x satisfied: chi2={chi2:.2f}, dof={dof}, p={p:.3f} -> {'associated' if p<0.05 else 'no clear association'}")
# Q: does wait time differ by clinic? 3 groups numeric (skewed) -> ANOVA & Kruskal
groups=[g.wait_minutes.values for _,g in d.groupby("clinic")]
F,pf=stats.f_oneway(*groups); H,pk=stats.kruskal(*groups)
print(f"[ANOVA] wait by clinic: F={F:.2f}, p={pf:.3f}")
print(f"[Kruskal] wait by clinic: H={H:.2f}, p={pk:.3f}  (skewed -> trust the rank test)")

[chi-square] treatment x satisfied: chi2=1.22, dof=1, p=0.269 -> no clear association
[ANOVA] wait by clinic: F=0.53, p=0.588
[Kruskal] wait by clinic: H=2.20, p=0.333  (skewed -> trust the rank test)


Not every test fires: satisfaction is <em>not</em> clearly tied to treatment here (p &#8776; 0.27), and wait time does <em>not</em> differ by clinic (p &#8776; 0.59). That is a feature, not a failure, the framework tells you which test to run; the data tell you the answer, and "no significant difference" is a legitimate, common result.

<div style="background:#ecfdf5;border-left:5px solid #059669;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#047857;letter-spacing:1px">REAL-WORLD EXAMPLE · ONE CLINIC DATASET, FOUR QUESTIONS</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">The decision map, applied end to end</div>
<div style="color:#4a5578;margin-top:6px"></div>
</div>

The clinic dataset (`ch81_clinic_visits.xlsx`) carries numeric, paired, and categorical columns at once, the perfect workout for the decision map. We pose four questions and let the framework choose the test for each.

In [5]:
rows=[]
rows.append(("Did BP drop within patients?","paired numeric","paired t-test",f"p={stats.ttest_rel(d.sysbp_after,d.sysbp_before).pvalue:.1e}","YES"))
rows.append(("New vs standard BP drop?","2 indep. numeric","two-sample (Welch) t",f"p={stats.ttest_ind(new,std,equal_var=False).pvalue:.1e}","YES"))
rows.append(("Satisfaction by treatment?","2 categoricals","chi-square",f"p={stats.chi2_contingency(pd.crosstab(d.treatment,d.satisfied))[1]:.2f}","no"))
rows.append(("Wait time by clinic?","3 groups numeric","ANOVA / Kruskal",f"p={stats.f_oneway(*groups).pvalue:.2f}","no"))
res=pd.DataFrame(rows, columns=["question","data shape","test chosen","result","significant?"])
print(res.to_string(index=False))

                    question       data shape          test chosen    result significant?
Did BP drop within patients?   paired numeric        paired t-test p=3.7e-48          YES
    New vs standard BP drop? 2 indep. numeric two-sample (Welch) t p=1.5e-09          YES
  Satisfaction by treatment?   2 categoricals           chi-square    p=0.27           no
        Wait time by clinic? 3 groups numeric      ANOVA / Kruskal    p=0.59           no


One spreadsheet, four questions, four correctly chosen tests, and two of them come back non-significant. That is exactly how real analysis goes: the hard part is rarely the arithmetic, it is <strong>matching the test to the question and the data shape</strong>, and then reporting honestly whether the effect is there. The new treatment genuinely lowers blood pressure more than standard; satisfaction and wait time show no reliable differences here.

<div style="background:#ffffff;border:1px solid #e6e9f2;border-radius:16px;padding:22px 26px;font-family:Inter,sans-serif">
<div style="font-size:19px;font-weight:800;color:#1a2138">✅ Choosing the right test</div>
<div style="color:#4a5578;line-height:1.8;margin-top:8px">Four questions pick the test: outcome type (numeric vs categorical), number of groups (1 / 2 / 3+), design (independent vs paired), and whether assumptions hold (else go nonparametric). Numeric &#8594; t / ANOVA (or Mann-Whitney / Kruskal-Wallis); categorical &#8594; z for proportions / chi-square. On one clinic dataset the new treatment beats standard on BP drop, while satisfaction and wait time show no reliable difference. Next, the capstone of the Part: A/B testing and online experiments.</div></div>

---
<div style="text-align:center;color:#8b94b3;font-size:12px;margin-top:10px">Statistics, Data Science and AI: A Visual Handbook · © 2026 John Fisher</div>